<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_22_encapsulation_scope/note_lesson_22_encapsulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 22 — Інкапсуляція, область видимості: замовлення, яке себе захищає

Бухгалтер знайшла в звіті від'ємний виторг: скрипт знижок писав прямо в `order.bill`. А ще виставляв статус `"delivered"` замовленням, які кухня й не починала. Перевірка в `__init__` захищає лише **народження** об'єкта. Сьогодні — як зробити, щоб у стан вели тільки **правильні двері**.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | інваріант і як його зламати; картка класу | 1.1–1.2 | |
| 2 | `_name` і `__name`: домовленість і name mangling, колізія імен у нащадках | 2.1–2.6 | |
| 3 | методи — двері в стан; «Tell, don't ask»; спершу перевір — потім змінюй | 3.1–3.3 | |
| 4 | статуси як скінченний автомат: `TRANSITIONS` і один `_move` | 4.1–4.4 | |
| 5 | `@property`: лише читання, сеттер з перевіркою, обчислювана властивість | 5.1–5.5 | **Задача 1** |
| 6 | усе разом: `SafeOrder` | 6.1–6.2 | **Задача 2** |
| 7 | область видимості: `global`, `nonlocal`, константи модуля, `_helper` і `__all__` | 7.1–7.8 | |
| 8 | де живе правило: у моделі чи в сервісі | 8.1 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`class`, `def`, константи) лише створює клас чи функцію — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу скажи, що буде, потім розгортай відповідь.
- Задач три. Розв'язок пишеш у клітинці з визначенням, перевірка — у наступній клітинці; вона друкує «✅ Задача N розв'язана».
- Теорія, діаграма автомата статусів і архітектура «інтерфейс проти стану» — у книзі: [Урок 22. Інкапсуляція, область видимості](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_22/).

## 🔁 Пригадай (без підглядання)

1. Коли виконується перевірка в `__init__`?
2. Навіщо `nonlocal`?
3. Що таке принцип підстановки Лісков?

<details>
<summary>Відповіді</summary>

1. Один раз — при створенні об'єкта.
2. Щоб змінити змінну оточуючої функції.
3. Нащадок має працювати всюди, де очікують батька.

</details>

---
## 1. Інваріант і як його зламати

**Інваріант** — правило, яке має бути правдою **завжди**, протягом усього життя об'єкта:

- сума чека більша за 0;
- статус змінюється лише за маршрутом «нове → готується → в дорозі → доставлено»;
- кількість використань промокоду не від'ємна.

### Приклад 1.1. Перевірка в `__init__` — і обхід за один рядок

`Order` з уроку 20: сума перевіряється при створенні. Клітинка нижче лише визначає клас.

In [ ]:
class Order:
    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self.bill = bill

**🔮 Прогноз:** що надрукує наступна клітинка — помилку чи число?

In [ ]:
order = Order(1, 540.0)
order.bill = order.bill - 600
print(order.bill)

<details>
<summary>Відповідь</summary>

`-60.0` — без жодної помилки: `bill` відкритий, писати в нього може будь-хто.

</details>

### Приклад 1.2. Народження захищене, життя — ні

Створити замовлення на −120 грн не вийде: спрацює перевірка `__init__`. А після створення в об'єкт можна записати що завгодно — навіть атрибут, якого в класі немає.

In [ ]:
try:
    Order(2, -120.0)
except ValueError as error:
    print("ValueError:", error)

order = Order(3, 320.0)
order.status = "delivered"     # такого атрибута в класі немає — Python мовчки створить новий
order.bill = "дорого"          # і тип ніхто не перевіряє
print(order.__dict__)

Перший рядок виводу — перевірка спрацювала при створенні. Другий — словник атрибутів після двох присвоєнь: `bill` став рядком, а `status` з'явився нізвідки. Жодного попередження: **відкритий атрибут — відкриті двері**.

### Картка класу: замовлення, яке себе захищає

Перш ніж писати код, спроєктуємо об'єкт (та сама картка, що в уроці 20). Новий рядок у ній — **правила**: інваріанти, які об'єкт має гарантувати сам.

| | `Order` |
|---|---|
| **1. Що це за річ?** | замовлення кафе, що проходить шлях від кухні до клієнта |
| **2. Що знає?** | номер `id`, суму чека, статус, історію статусів — **внутрішній** стан |
| **3. Що вміє?** | дати знижку, перейти в наступний статус (`cook`, `send`, `deliver`, `cancel`), показати стан для читання |
| **4. Які правила?** | сума > 0; знижка від 1 до 50 % і лише новому замовленню; статус — лише за дозволеними переходами |
| **5. Чого НЕ робить?** | не видає собі номер, не знає про водіїв і чужі доставки — це справа сервісу (розділ 8) |

```text
зовнішній код            двері (публічні методи)        правила                 стан
сервіс, звіт, скрипт  →  apply_discount, cook, send  →  перевірки, _move  →  _bill, _status, _history
        │                                                                          ▲
        └──────────────── order._status = ...  (обхідний шлях) ────────────────────┘
```

Сьогодні крок за кроком закриваємо обхідний шлях: спершу домовленістю в іменах, потім методами, автоматом і `@property`.

---
## 2. `_name` і `__name`

Python не має ключових слів `private` чи `public`. Замість них — **домовленості в іменах**:

| Ім'я | Що означає | Чи захищає |
|---|---|---|
| `bill` | публічний атрибут: частина інтерфейсу | ні |
| `_bill` | внутрішній: «не чіпай ззовні, це деталь реалізації» | домовленість, Python не заважає |
| `__bill` | Python перейменовує в `_Order__bill` (name mangling) | від випадкових збігів імен у нащадках |

### Приклад 2.1. Що насправді лежить в об'єкті

Одне поле з одним підкресленням, одне — з двома.

In [ ]:
class Order:
    def __init__(self, order_id, bill):
        self.id = order_id
        self._bill = bill
        self.__kitchen_note = "без цибулі"

    def kitchen_note(self):
        return self.__kitchen_note       # усередині класу — звичайне ім'я

**🔮 Прогноз:** як виглядатиме `order.__dict__`? Чи вдасться прочитати `order.__kitchen_note`?

In [ ]:
order = Order(1, 540.0)
print(order.__dict__)
print(order._bill)
print(order._bill - 600)          # одне підкреслення нічого не забороняє
print(order.kitchen_note())       # метод класу бачить поле

try:
    print(order.__kitchen_note)
except AttributeError as error:
    print("AttributeError:", error)

<details>
<summary>Відповідь</summary>

`__kitchen_note` став `_Order__kitchen_note` — name mangling. `_bill` — звичайний атрибут, підкреслення лише домовленість.

</details>

### Приклад 2.2. Name mangling — не замок

Поле з двома підкресленнями доступне за **повним** іменем. Це захист від випадковостей, а не від того, хто свідомо лізе всередину.

In [ ]:
print(order._Order__kitchen_note)
order._Order__kitchen_note = "з подвійною цибулею"
print(order.kitchen_note())

Прочитали й переписали «приховане» поле. Але рядок `order._Order__kitchen_note = ...` неможливо написати **випадково** — його одразу видно на рев'ю.

### Приклад 2.3. Перейменування діє лише в тілі класу

Python перейменовує `__name` тоді, коли **компілює тіло класу**. Поза класом `__name` — звичайне ім'я.

In [ ]:
order.__secret = "код домофону 1234"     # присвоєння ПОЗА класом
print(order.__dict__)
print(order.__secret)

У словнику з'явився ключ `'__secret'` без імені класу: поза класом перейменування не відбувається, і такий атрибут нічим не відрізняється від звичайного. Отже, `__` має сенс лише всередині `class`.

### Приклад 2.4. Колізія імен у нащадках

Навіщо тоді `__`? Ось випадковість, від якої воно захищає. Батько тримає статус у `_status`, а автор нащадка, не знаючи цього, назвав так своє поле.

In [ ]:
class Delivery:
    def __init__(self, order_id):
        self.order_id = order_id
        self._status = "нова"

    def status(self):
        return self._status


class TrackedDelivery(Delivery):
    def __init__(self, order_id, gps):
        super().__init__(order_id)
        self._status = f"GPS {gps}"

**🔮 Прогноз:** що поверне `tracked.status()`?

In [ ]:
tracked = TrackedDelivery(7, "50.45,30.52")
print(tracked.status())
print(tracked.__dict__)

<details>
<summary>Відповідь</summary>

`GPS 50.45,30.52` — нащадок затер `_status` батька. Виправлення — `__status`: у кожного класу своє поле.

</details>

### Приклад 2.5. Два підкреслення — у кожного класу своє поле

Те саме, але поле названо `__status` і в батька, і в нащадка.

In [ ]:
class Delivery:
    def __init__(self, order_id):
        self.order_id = order_id
        self.__status = "нова"

    def status(self):
        return self.__status


class TrackedDelivery(Delivery):
    def __init__(self, order_id, gps):
        super().__init__(order_id)
        self.__status = f"GPS {gps}"

In [ ]:
tracked = TrackedDelivery(7, "50.45,30.52")
print(tracked.status())
print(tracked.__dict__)

`status()` повертає «нова»: метод батька читає `_Delivery__status`, а нащадок записав своє `_TrackedDelivery__status`. Два різні атрибути — і жоден не затер інший.

### Приклад 2.6. Ціна `__`: нащадок не бачить поле батька

Зворотний бік: нащадок, який **хоче** прочитати поле батька, теж його не знайде — `self.__status` у тілі нащадка стає `self._LoudDelivery__status`.

In [ ]:
class LoudDelivery(Delivery):
    def shout(self):
        return self.__status.upper()          # шукає _LoudDelivery__status

    def shout_ok(self):
        return self.status().upper()          # просимо через публічний метод батька

In [ ]:
loud = LoudDelivery(8)
try:
    loud.shout()
except AttributeError as error:
    print("AttributeError:", error)
print(loud.shout_ok())

Нащадок дістає стан батька лише через **інтерфейс** — `status()`. Тому `__` беруть не за замовчуванням.

> **Коли яке підкреслення.** За замовчуванням — одне: `_bill`. Воно чесно каже «це внутрішнє» і не заважає нащадкам. Два — лише для атрибутів класу, який **задумано** як базовий для чужих нащадків, щоб їхні поля випадково не затерли твої ([PEP 8](https://peps.python.org/pep-0008/#designing-for-inheritance)).

---
## 3. Методи — двері в стан

Сховати `bill` за підкресленням мало: треба дати **правильний спосіб** її змінити. Знижку дає не той, хто викликає, віднімаючи число, а сам об'єкт — методом з перевіркою.

### Приклад 3.1. Знижка — метод з правилом

In [ ]:
class Order:
    MAX_DISCOUNT = 50

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill

    def bill(self):
        return self._bill

    def apply_discount(self, percent):
        if not 0 < percent <= self.MAX_DISCOUNT:
            raise ValueError(f"знижка має бути від 1 до {self.MAX_DISCOUNT} %, а маємо {percent}")
        self._bill = round(self._bill * (100 - percent) / 100, 2)

In [ ]:
order = Order(1, 540.0)
order.apply_discount(10)
print(order.bill())

try:
    order.apply_discount(111)
except ValueError as error:
    print(error)
print(order.bill())

Знижка 10 % пройшла (486.0), знижку 111 % відхилено, і сума лишилась 486.0. Правило «не більше 50 %» живе в одному місці.

### Приклад 3.2. «Ask» проти «Tell»

Скрипт знижок з початку уроку **питав** об'єкт про суму, рахував сам і записував назад. Порівняймо з тим, коли ми **кажемо** об'єкту, що зробити.

In [ ]:
def discount_script(order, percent):
    """«Ask»: читаємо стан, рахуємо самі, записуємо назад — правило знову поза об'єктом."""
    order._bill = order.bill() * (100 - percent) / 100

In [ ]:
asked = Order(2, 540.0)
discount_script(asked, 111)
print("ask: ", asked.bill())

told = Order(3, 540.0)
try:
    told.apply_discount(111)
except ValueError as error:
    print("tell:", error)
print("tell:", told.bill())

Версія «ask» дала від'ємний чек: вона писала в `_bill` в обхід правила (і підкреслення її не зупинило — лише зробило порушення помітним). Версія «tell» — `apply_discount` — відхилила знижку. Принцип **«Tell, don't ask»**: кажи об'єкту, що зробити, замість того щоб читати його стан, рахувати й записувати назад.

### Приклад 3.3. Спершу перевір — потім змінюй

Метод-двері має сенс, лише якщо після помилки стан **лишається правильним**. Ось поспішна версія, яка спершу змінює суму, а потім перевіряє.

In [ ]:
class HastyOrder(Order):
    def apply_discount(self, percent):
        self._bill = round(self._bill * (100 - percent) / 100, 2)    # спершу змінили...
        if not 0 < percent <= self.MAX_DISCOUNT:                     # ...потім перевірили
            raise ValueError(f"знижка {percent} % заборонена")

In [ ]:
hasty = HastyOrder(4, 540.0)
try:
    hasty.apply_discount(111)
except ValueError as error:
    print(error)
print(hasty.bill())

Виняток є, але сума вже від'ємна: зміна сталася **до** перевірки. Правило для кожного методу-дверей: **спочатку всі перевірки, потім зміна стану**. Так і зроблено в `Order.apply_discount` з прикладу 3.1.

---
## 4. Статуси: скінченний автомат

Статус — найнебезпечніше поле: від нього залежать кухня, водії й звіти. Замовлення проходить чіткий маршрут, і не кожен перехід дозволено:

```text
           cook()             send()               deliver()
   new ───────────► cooking ───────────► on_the_way ───────────► delivered
    │                  │
    │ cancel()         │ cancel()
    ▼                  ▼
 cancelled ◄───────────┘
```

Скасувати можна, поки водій не виїхав. `delivered` і `cancelled` — кінцеві стани. Така схема — **скінченний автомат** (finite state machine). У код вона переноситься майже дослівно: словник «стан → дозволені наступні стани» і **один** внутрішній метод `_move`, через який проходить кожна зміна.

### Приклад 4.1. Автомат статусів

In [ ]:
class Order:
    TRANSITIONS = {
        "new": {"cooking", "cancelled"},
        "cooking": {"on_the_way", "cancelled"},
        "on_the_way": {"delivered"},
        "delivered": set(),
        "cancelled": set(),
    }

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill
        self._status = "new"
        self._history = ["new"]

    def _move(self, new_status):
        if new_status not in self.TRANSITIONS[self._status]:
            raise ValueError(f"замовлення №{self.id}: не можна {self._status} → {new_status}")
        self._status = new_status
        self._history.append(new_status)

    def cook(self):
        self._move("cooking")

    def send(self):
        self._move("on_the_way")

    def deliver(self):
        self._move("delivered")

    def cancel(self):
        self._move("cancelled")

In [ ]:
order = Order(1, 540.0)
order.cook()
order.send()
try:
    order.cancel()
except ValueError as error:
    print(error)
order.deliver()
print(order._history)

Скасування в дорозі відхилено, і стан лишився правильним. Публічні методи `cook`, `send`, `deliver`, `cancel` — **інтерфейс** замовлення. `_move`, `_status` і `_history` — внутрішня кухня.

### Приклад 4.2. Правила — це дані

Оскільки правила лежать у словнику, їх можна надрукувати як таблицю — без жодного об'єкта.

In [ ]:
for state, next_states in Order.TRANSITIONS.items():
    print(f"{state:>10} → {', '.join(sorted(next_states)) or '(кінцевий стан)'}")

П'ять рядків — увесь автомат. Якби правила були розкидані `if`-ами по методах, таку таблицю довелося б складати вручну, читаючи код.

### Приклад 4.3. Які дії дозволені зараз

Ті самі дані відповідають і на питання інтерфейсу: які кнопки показати оператору для цього замовлення? Додамо словник `ACTIONS` (дія → стан, у який вона веде) і метод `allowed_actions()`.

> **`class Order(Order)`.** Тут і далі ми нарощуємо клас частинами, щоб не повторювати весь код: новий `Order` наслідує попередній і додає методи. У справжньому проєкті все це живе в одному класі.

In [ ]:
class Order(Order):
    ACTIONS = {"cook": "cooking", "send": "on_the_way", "deliver": "delivered", "cancel": "cancelled"}

    def allowed_actions(self):
        allowed = self.TRANSITIONS[self._status]
        return [action for action, target in self.ACTIONS.items() if target in allowed]

In [ ]:
order = Order(2, 320.0)
for step in ("cook", "send", "deliver"):
    print(f"{order._status:>10}: можна {order.allowed_actions()}")
    getattr(order, step)()                 # викликаємо метод за назвою: order.cook() тощо
print(f"{order._status:>10}: можна {order.allowed_actions()}")

На кожному кроці список дій звужується, а в `delivered` він порожній. Метод не містить жодного `if` про конкретні статуси — лише читає `TRANSITIONS`.

### Приклад 4.4. Новий стан — рядок у словнику

Кафе запускає самовивіз: зготоване замовлення може стати `ready` («чекає на клієнта»), а потім `picked_up`. Метод `_move` не змінюється — змінюється лише словник.

In [ ]:
class PickupOrder(Order):
    TRANSITIONS = {
        **Order.TRANSITIONS,
        "cooking": {"on_the_way", "ready", "cancelled"},
        "ready": {"picked_up"},
        "picked_up": set(),
    }

    def ready(self):
        self._move("ready")

    def pick_up(self):
        self._move("picked_up")

In [ ]:
pickup = PickupOrder(3, 210.0)
pickup.cook()
pickup.ready()
pickup.pick_up()
print(pickup._history)

try:
    pickup.deliver()
except ValueError as error:
    print(error)

`{**Order.TRANSITIONS, ...}` — копія батьківських правил, у якій перезаписано рядок `"cooking"` і додано два нових стани. Доставити вже видане замовлення не можна: перевірку зробив той самий `_move` з прикладу 4.1.

---
## 5. `@property`: читати — так, писати — лише через правила

Читати `order._history` ззовні — незручно й непристойно (підкреслення!). Хочеться `order.status`, як звичайний атрибут, але без запису. Для цього є **`@property`**: метод, до якого звертаються як до атрибута.

### Приклад 5.1. Властивості лише для читання

In [ ]:
class Order(Order):
    @property
    def status(self):
        return self._status

    @property
    def history(self):
        return tuple(self._history)

In [ ]:
order = Order(2, 320.0)
order.cook()
print(order.status, order.history)

try:
    order.status = "delivered"
except AttributeError as error:
    print(type(error).__name__, "—", error)

`order.status` — без дужок, а насправді викликає метод. Сеттера немає — запис дає `AttributeError`. Статус змінюється лише методами автомата.

### Приклад 5.2. Знайди помилку: властивість віддає сам список

Чому `history` повертає `tuple(...)`, а не просто `self._history`? Ось версія з помилкою.

In [ ]:
class LeakyOrder(Order):
    @property
    def history(self):
        return self._history              # віддає сам список — у чому проблема?

In [ ]:
leaky = LeakyOrder(9, 300.0)
leaky.history.append("delivered")         # дописали в історію ззовні
print(leaky.history, leaky.status)

safe = Order(10, 300.0)
try:
    safe.history.append("delivered")
except AttributeError as error:
    print("AttributeError:", error)
assert safe.history == ("new",) and safe.status == "new"
print(safe.history, safe.status)

`LeakyOrder` віддав **сам** список: історія тепер каже «delivered», а статус — «new». Автомат оминули, не торкнувшись жодного поля з підкресленням. `Order` віддає кортеж-**копію**: у неї нема `append`, а якби й було — змінилась би копія, не стан.

### Приклад 5.3. Сеттер: запис дозволено, але з перевіркою

Якщо запис дозволений, але з правилами, до `@property` додають **сеттер**. Тоді звичайне присвоєння запускає перевірку.

In [ ]:
class Order(Order):
    @property
    def bill(self):
        return self._bill

    @bill.setter
    def bill(self, value):
        if value <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {value}")
        self._bill = value

In [ ]:
order = Order(3, 980.0)
order.bill = 900.0
try:
    order.bill = order.bill - 1000
except ValueError as error:
    print(error)
print(order.bill)

Той самий рядок, що на початку уроку зламав звіт, тепер зупиняється з поясненням. Зовнішній код не змінився — `order.bill = …`, — а інваріант захищено.

### Приклад 5.4. Обчислювана властивість

Властивість не зобов'язана щось зберігати: вона може **обчислювати** відповідь зі стану. `is_final` — «замовлення вже нікуди не рухається» — випливає з `TRANSITIONS`; окреме поле для нього лише могло б розійтися зі статусом.

In [ ]:
class Order(Order):
    @property
    def is_final(self):
        return not self.TRANSITIONS[self._status]

In [ ]:
order = Order(4, 450.0)
for step in (order.cook, order.send, order.deliver):
    print(f"{order.status:>10}: кінцевий? {order.is_final}")
    step()
print(f"{order.status:>10}: кінцевий? {order.is_final}")

try:
    order.is_final = False
except AttributeError:
    print("is_final лише для читання")

`True` лише в `delivered`: порожня множина дозволених переходів — це «кінець маршруту». Записати `is_final` не можна — сеттера немає.

### Приклад 5.5. Де живе властивість

`@property` — не поле об'єкта, а об'єкт **класу**. В екземплярі лежить лише `_bill`.

In [ ]:
order = Order(5, 700.0)
print(order.__dict__)
print(type(Order.bill).__name__, type(Order.status).__name__)
print("bill" in order.__dict__)

У словнику екземпляра — `id`, `_bill`, `_status`, `_history`; імені `bill` там немає. `Order.bill` — об'єкт типу `property`, що живе в класі: коли пишемо `order.bill`, Python знаходить його в класі й викликає метод-читач. Докладно про `@property` і дескриптори — в уроці 24.

### Задача 1. Захищений промокод

Спершу — картка класу:

| | `Promo` |
|---|---|
| **1. Що це?** | промокод на знижку з обмеженою кількістю використань |
| **2. Що знає?** | `code`, `percent` — публічно; залишок використань — у **`__left`** (name mangling) |
| **3. Що вміє?** | `apply(bill)` — повертає суму зі знижкою `round(bill * (100 - percent) / 100, 2)` і зменшує залишок; `left` — властивість **лише для читання** |
| **4. Правила** | залишок не від'ємний: вичерпаний промокод → `ValueError` |
| **5. Не робить** | не змінює сам чек замовлення — лише рахує нову суму |

Нащадок `VipPromo` (його вже написано) зберігає VIP-бонуси у **своєму** `__left`. Якщо твій `Promo` використовує `__left`, поле нащадка не затре батьківське (приклад 2.5).

Перевірка — у наступній клітинці.

In [ ]:
class Promo:
    # BEGIN SOLUTION
    def __init__(self, code, percent, uses):
        self.code = code
        self.percent = percent
        self.__left = uses

    @property
    def left(self):
        return self.__left

    def apply(self, bill):
        if self.__left == 0:
            raise ValueError(f"промокод {self.code} вичерпано")
        self.__left -= 1
        return round(bill * (100 - self.percent) / 100, 2)
    # END SOLUTION


class VipPromo(Promo):
    def __init__(self, code, percent, uses, bonus_uses):
        super().__init__(code, percent, uses)
        self.__left = bonus_uses          # своє поле нащадка

In [ ]:
lucky = Promo("LUCKY20", 20, 2)
assert lucky.apply(500.0) == 400.0 and lucky.left == 1
assert lucky.apply(250.0) == 200.0 and lucky.left == 0
try:
    lucky.apply(100.0)
except ValueError:
    pass
else:
    raise AssertionError("вичерпаний промокод — ValueError")
try:
    lucky.left = 999
except AttributeError:
    pass
else:
    raise AssertionError("left — лише для читання")
assert "_Promo__left" in lucky.__dict__, "лічильник має жити в __left"
vip = VipPromo("VIP30", 30, 3, 10)
assert vip.left == 3 and vip._VipPromo__left == 10, "поле нащадка не затерло батьківське"
print("✅ Задача 1 розв'язана")

---
## 6. Усе разом: `SafeOrder`

Зберемо в один клас усе з розділів 3–5: знижку з двома правилами, автомат статусів і три властивості лише для читання.

### Приклад 6.1. Замовлення, яке себе захищає

In [ ]:
class SafeOrder:
    TRANSITIONS = Order.TRANSITIONS
    MAX_DISCOUNT = 50

    def __init__(self, order_id, bill):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        self.id = order_id
        self._bill = bill
        self._status = "new"
        self._history = ["new"]

    @property
    def bill(self):
        return self._bill

    @property
    def status(self):
        return self._status

    @property
    def history(self):
        return tuple(self._history)

    def apply_discount(self, percent):
        if self._status != "new":
            raise ValueError(f"знижку можна дати лише новому замовленню, а статус {self._status}")
        if not 0 < percent <= self.MAX_DISCOUNT:
            raise ValueError(f"знижка має бути від 1 до {self.MAX_DISCOUNT} %, а маємо {percent}")
        self._bill = round(self._bill * (100 - percent) / 100, 2)

    def _move(self, new_status):
        if new_status not in self.TRANSITIONS[self._status]:
            raise ValueError(f"замовлення №{self.id}: не можна {self._status} → {new_status}")
        self._status = new_status
        self._history.append(new_status)

    def cook(self):
        self._move("cooking")

    def send(self):
        self._move("on_the_way")

    def deliver(self):
        self._move("delivered")

    def cancel(self):
        self._move("cancelled")

Що в ключових рядках:

- `TRANSITIONS = Order.TRANSITIONS` — правила переходів беремо з автомата розділу 4: дані, а не розкидані `if`;
- `bill`, `status`, `history` — властивості лише для читання; `history` віддає копію;
- `apply_discount` перевіряє **два** правила — статус і розмір знижки — і лише потім змінює суму;
- `_move` — єдине місце, де змінюється статус.

### Приклад 6.2. Три спроби порушити правила

Пробуємо: знижку після початку готування, доставку без відправлення і пряме присвоєння суми.

In [ ]:
order = SafeOrder(1, 540.0)
order.apply_discount(10)
order.cook()
for action in (lambda: order.apply_discount(5), order.deliver, lambda: setattr(order, "bill", 1.0)):
    try:
        action()
    except (ValueError, AttributeError) as error:
        print(type(error).__name__, "—", error if isinstance(error, ValueError) else "запис заборонено")
order.send()
order.deliver()
print(order.bill, order.status, order.history)

Три рядки помилок — три відхилені спроби. Останній рядок — стан після чесного маршруту: сума 486.0 (знижка 10 % до готування), статус `delivered` і правдива історія.

### Задача 2. Повернення замовлення

Клієнт може **повернути** доставлене замовлення. Допиши `ReturnableOrder(SafeOrder)`:

- новий стан `"returned"`, перехід лише з `"delivered"` — це рядки в `TRANSITIONS` (`{**SafeOrder.TRANSITIONS, ...}`, як у прикладі 4.4), а не новий `if` у `_move`;
- `__init__` викликає батьківський і додає `self._return_reason = None`;
- метод `return_order(reason)`: порожня причина (зокрема з самих пробілів) → `ValueError`; інакше `_move("returned")` і збереження причини в `_return_reason`;
- властивість `return_reason` лише для читання.

```text
order.return_order("холодна піца")
order.status         →  'returned'
order.return_reason  →  'холодна піца'
order.history[-2:]   →  ('delivered', 'returned')
```

Перевірка — у наступній клітинці.

In [ ]:
class ReturnableOrder(SafeOrder):
    # BEGIN SOLUTION
    TRANSITIONS = {**SafeOrder.TRANSITIONS, "delivered": {"returned"}, "returned": set()}

    def __init__(self, order_id, bill):
        super().__init__(order_id, bill)
        self._return_reason = None

    @property
    def return_reason(self):
        return self._return_reason

    def return_order(self, reason):
        if not reason.strip():
            raise ValueError("вкажи причину повернення")
        self._move("returned")
        self._return_reason = reason
    # END SOLUTION

In [ ]:
order = ReturnableOrder(5, 700.0)
order.cook(); order.send(); order.deliver()
order.return_order("холодна піца")
assert order.status == "returned" and order.return_reason == "холодна піца"
assert order.history[-2:] == ("delivered", "returned")
for bad in (lambda: ReturnableOrder(6, 100.0).return_order("інше"),
            lambda: order.return_order("ще раз"),
            lambda: setattr(order, "return_reason", "інше")):
    try:
        bad()
    except (ValueError, AttributeError):
        pass
    else:
        raise AssertionError("очікували ValueError чи AttributeError")
fresh = ReturnableOrder(7, 100.0)
fresh.cook(); fresh.send(); fresh.deliver()
try:
    fresh.return_order("   ")
except ValueError:
    pass
else:
    raise AssertionError("порожня причина — ValueError")
assert fresh.status == "delivered" and fresh.return_reason is None, "після помилки стан не змінюється"
print("✅ Задача 2 розв'язана")

---
## 7. Область видимості: чим менше, тим безпечніше

Інкапсуляція класу — окремий випадок ширшої ідеї: **кожна змінна має бути видимою в найменшій потрібній області**. В уроці 19 ми розібрали LEGB — як Python **шукає** ім'я. Тепер — де його **тримати**.

### Приклад 7.1. Присвоєння робить змінну локальною

Перша версія сервісу рахувала замовлення в глобальній змінній.

In [ ]:
total_orders = 0


def add_order_broken():
    total_orders += 1

In [ ]:
try:
    add_order_broken()
except UnboundLocalError as error:
    print(type(error).__name__, "—", error)

Та сама пастка, що з `nonlocal` в уроці 19: присвоєння `+=` усередині функції робить `total_orders` **локальною** змінною, а локальна ще не має значення.

### Приклад 7.2. «Виправлення» через `global`

In [ ]:
def add_order():
    global total_orders
    total_orders += 1
    return total_orders

In [ ]:
add_order()
add_order()
print(total_orders)

Працює: 2. Але тепер змінити `total_orders` може **будь-яка** функція будь-якого модуля.

### Приклад 7.3. Два відділення — один лічильник

Кафе відкриває друге відділення. Обидва використовують `add_order()`.

**🔮 Прогноз:** які номери отримають перші замовлення Подолу й Оболоні?

In [ ]:
podil_first = add_order()
obolon_first = add_order()
print(podil_first, obolon_first)

<details>
<summary>Відповідь</summary>

`3 4`: лічильник один на весь модуль, і відділення рахують у ту саму змінну. Щоб знайти, хто і де її змінює, треба прочитати **весь** код. Рішення — стан в об'єкті (приклад 7.4) або в замиканні (приклад 7.5).

</details>

### Приклад 7.4. Лічильник в об'єкті

In [ ]:
class Branch:
    def __init__(self, name):
        self.name = name
        self._next_id = 1

    def add_order(self):
        order_id = self._next_id
        self._next_id += 1
        return order_id

In [ ]:
podil, obolon = Branch("Поділ"), Branch("Оболонь")
print(podil.add_order(), podil.add_order(), obolon.add_order())

`1 2 1`: у кожного відділення свій `_next_id`, і змінити його можна лише методом `add_order`.

### Приклад 7.5. Лічильник у замиканні (`nonlocal`)

Якщо від об'єкта потрібна лише одна дія, вистачить замикання з уроку 19. Змінна `count` живе в оточуючій функції й недоступна ззовні за іменем узагалі.

In [ ]:
def make_counter(prefix):
    count = 0

    def next_id():
        nonlocal count            # без nonlocal — той самий UnboundLocalError, що в прикладі 7.1
        count += 1
        return f"{prefix}-{count}"

    return next_id

In [ ]:
podil_next = make_counter("П")
obolon_next = make_counter("О")
print(podil_next(), podil_next(), obolon_next(), podil_next())

Кожен виклик `make_counter` створює **свою** змінну `count`: у Подолу дійшло до 3, в Оболоні — 1. Жоден інший код не може її змінити — ні через `global`, ні через атрибут.

### Приклад 7.6. Читати змінну модуля можна без `global`

`global` потрібен лише для **присвоєння**. Читати константу модуля функція може й так. А присвоєння без `global` створює нову локальну змінну, яка ховає модульну.

In [ ]:
SERVICE_FEE = 30              # константа модуля: великі літери — «не змінюється»


def fee_for(bill):
    return SERVICE_FEE if bill < 500 else 0      # лише читає — шукає за LEGB у модулі


def fee_for_vip(bill):
    SERVICE_FEE = 0                              # нова ЛОКАЛЬНА змінна
    return SERVICE_FEE

In [ ]:
print(fee_for(320), fee_for(700), fee_for_vip(320))
print("SERVICE_FEE після всіх викликів:", SERVICE_FEE)

`fee_for_vip` не зачепив модульну константу: його `SERVICE_FEE` — локальна змінна, що існує лише під час виклику.

### Приклад 7.7. Змінний список модуля — пастка без `global`

`global` не потрібен і тоді, коли ми **змінюємо об'єкт**, а не присвоюємо ім'я: `append` — не присвоєння. Тому змінний список на рівні модуля можна зіпсувати звідусіль.

In [ ]:
ORDERS_LOG = []


def log_order(order_id):
    ORDERS_LOG.append(order_id)        # змінюємо об'єкт — global не потрібен


def nightly_cleanup():
    ORDERS_LOG.clear()                 # «прибирання» з іншого кінця програми

In [ ]:
log_order(1)
log_order(2)
print(ORDERS_LOG)
nightly_cleanup()
log_order(3)
print(ORDERS_LOG)

Журнал втратив замовлення 1 і 2: його очистила функція, про яку автор `log_order` міг і не знати. Це той самий спільний стан, що й з `global`, лише без попереджувального слова.

| Де тримати змінну | Хто може змінити | Коли |
|---|---|---|
| локальна змінна функції | лише ця функція | проміжні обчислення — за замовчуванням |
| змінна замикання (`nonlocal`) | лише вкладена функція | маленький стан однієї дії: лічильник |
| атрибут об'єкта `self._x` | методи об'єкта | стан, що живе між викликами |
| змінна модуля | будь-який код, що імпортує модуль | константи: `SERVICE_FEE`, `MAX_DISCOUNT` — **не змінюються** |
| `global` | будь-хто | майже ніколи |

### Приклад 7.8. Межа модуля: `_helper` і `__all__`

Ті самі домовленості діють для модулів з уроку 13. Клітинка з `%%writefile` створює файл `pricing.py` поруч з ноутбуком (і в Jupyter, і в Colab).

In [ ]:
%%writefile pricing.py
__all__ = ["fare_for"]

FEES = {"Поділ": 60, "Оболонь": 80}


def _round_to_ten(value):
    return round(value / 10) * 10


def fare_for(district, night=False):
    fare = FEES[district]
    return _round_to_ten(fare * 1.3) if night else fare

Тепер імпортуємо модуль і подивимось, що потрапляє в `from pricing import *`. Щоб не засмічувати простір імен ноутбука, виконуємо цей імпорт в окремому словнику `namespace`.

In [ ]:
import pricing

print(pricing.fare_for("Поділ"), pricing.fare_for("Поділ", night=True))
print(pricing._round_to_ten(87))      # доступна, але «не частина інтерфейсу»

namespace = {}
exec("from pricing import *", namespace)
print(sorted(name for name in namespace if not name.startswith("__")))

- нічний тариф Подолу: 60 · 1.3 = 78 → округлено до 80;
- `_round_to_ten` викликається за повним іменем `pricing._round_to_ten` — Python не забороняє, але підкреслення каже: «завтра її можуть перейменувати»;
- `from pricing import *` приніс лише `fare_for` — те, що перелічено в `__all__`. Без `__all__` прийшли б усі імена без підкреслення: `FEES` і `fare_for`.

---
## 8. Де живе правило: у моделі чи в сервісі

| Правило | Де | Чому |
|---|---|---|
| сума чека > 0 | `Order` | стосується лише одного об'єкта — його перевіряє сам об'єкт |
| переходи статусів | `Order` | стан одного замовлення |
| номери замовлень унікальні | `DeliveryService` | знає про всі замовлення одразу |
| один водій — одна активна доставка | `DeliveryService` | зв'язок між **різними** об'єктами: замовлення не знає про чужі доставки |

Просте правило: інваріант одного об'єкта — в його класі; правило між об'єктами — в тому, хто тримає їх усіх.

| | `DeliveryService` |
|---|---|
| **1. Що це?** | диспетчерська: тримає всі замовлення й доставки зміни |
| **2. Що знає?** | `_orders` (номер → замовлення), `_next_id`, `_deliveries` (номер замовлення → водій) |
| **3. Що вміє?** | `add_order(bill)` — створити замовлення з унікальним номером; `order(order_id)` — знайти; `orders`, `deliveries` — читання копій |
| **4. Правила** | номери унікальні; невідомий номер → `ValueError`; водій не везе два замовлення одночасно |
| **5. Не робить** | не змінює статус замовлення напряму — лише просить `order.send()`, і перевіряє автомат самого замовлення |

### Приклад 8.1. Сервіс видає номери

In [ ]:
class DeliveryService:
    def __init__(self):
        self._orders = {}
        self._next_id = 1
        self._deliveries = {}

    def add_order(self, bill):
        order = SafeOrder(self._next_id, bill)      # перевірку суми робить сама модель
        self._orders[order.id] = order
        self._next_id += 1
        return order

    def order(self, order_id):
        if order_id not in self._orders:
            raise ValueError(f"замовлення №{order_id} немає")
        return self._orders[order_id]

    @property
    def orders(self):
        return tuple(self._orders.values())

    @property
    def deliveries(self):
        return dict(self._deliveries)               # копія: зміни ззовні не зачеплять сервіс

In [ ]:
service = DeliveryService()
first = service.add_order(540.0)
second = service.add_order(320.0)
print([order.id for order in service.orders])

for bad in (lambda: service.add_order(-50.0), lambda: service.order(99)):
    try:
        bad()
    except ValueError as error:
        print("ValueError:", error)
print("замовлень після помилок:", len(service.orders), "| наступний номер:", service._next_id)

Номери 1 і 2 видав сервіс, а не самі замовлення. Від'ємну суму відхилила **модель** (`SafeOrder.__init__`) — сервіс її не перевіряє вдруге. Після помилок замовлень так само два, і номер 3 не «згорів»: `_next_id` збільшується лише після успішного створення.

### Задача 3. Водій не везе два замовлення одночасно

Допиши метод `add_delivery(order_id, driver)` у `DispatchService(DeliveryService)`. Порядок кроків важливий — **спершу всі перевірки, потім зміни** (приклад 3.3):

1. знайти замовлення через `self.order(order_id)` — для невідомого номера він сам кине `ValueError`;
2. якщо `driver` уже везе замовлення — тобто є запис у `self._deliveries` з цим водієм, і статус того замовлення `"on_the_way"`, — `ValueError`. Це правило **сервісу**: замовлення не знає про чужі доставки;
3. `order.send()` — перехід перевіряє **модель**: відправити можна лише те, що готується;
4. лише після успішного `send()` записати `self._deliveries[order_id] = driver`.

Коли замовлення доставлено (`order.deliver()`), водій знову вільний: його попередня доставка вже не `"on_the_way"`.

Перевірка — у наступній клітинці.

In [ ]:
class DispatchService(DeliveryService):
    def add_delivery(self, order_id, driver):
        # BEGIN SOLUTION
        order = self.order(order_id)
        busy = any(name == driver and self._orders[other].status == "on_the_way"
                   for other, name in self._deliveries.items())
        if busy:
            raise ValueError(f"водій {driver} уже везе замовлення")
        order.send()
        self._deliveries[order_id] = driver
        # END SOLUTION

In [ ]:
service = DispatchService()
first, second, third = service.add_order(540.0), service.add_order(320.0), service.add_order(150.0)
assert (first.id, second.id, third.id) == (1, 2, 3)
first.cook()
second.cook()
service.add_delivery(1, "Андрій")
assert first.status == "on_the_way" and service.deliveries == {1: "Андрій"}

for bad in (lambda: service.add_delivery(2, "Андрій"),     # водій зайнятий — правило сервісу
            lambda: service.add_delivery(3, "Олена"),      # ще не готується — правило моделі
            lambda: service.add_delivery(99, "Олена")):    # такого замовлення немає
    try:
        bad()
    except ValueError:
        pass
    else:
        raise AssertionError("очікували ValueError")
assert second.status == "cooking" and third.status == "new", "невдала спроба не змінює статус"
assert service.deliveries == {1: "Андрій"}, "невдала спроба нічого не записує"

first.deliver()
service.add_delivery(2, "Андрій")                          # тепер водій вільний
assert service.deliveries == {1: "Андрій", 2: "Андрій"}
service.deliveries[7] = "Хтось"
assert 7 not in service.deliveries, "deliveries віддає копію"
print("✅ Задача 3 розв'язана")

## ✅ Самоперевірка

1. Чому перевірки в `__init__` недостатньо?
2. Чим `_bill` відрізняється від `__bill`? Чи можна прочитати `__bill` ззовні?
3. Чому нащадок не бачить батьківського `self.__status`, і як йому дістати статус?
4. Чому метод-двері спершу перевіряє, а потім змінює стан?
5. Навіщо автомату словник `TRANSITIONS` і один `_move`?
6. Чому `history` повертає кортеж?
7. Чому `ORDERS_LOG.append(...)` працює без `global`, а `total_orders += 1` — ні?
8. Де жити правилу «один водій — одна активна доставка»?

<details>
<summary>Відповіді</summary>

1. `__init__` виконується один раз; далі відкритий атрибут змінює будь-хто.
2. `_bill` — домовленість; `__bill` → `_Order__bill` (name mangling). Ззовні — лише за повним іменем.
3. У тілі нащадка `self.__status` стає `self._Нащадок__status` — іншим атрибутом. Дістати можна через публічний метод батька, `status()`.
4. Інакше після винятку стан уже зіпсовано: виняток є, а сума від'ємна (приклад 3.3).
5. Правила — дані в одному місці, перевірка — в одному методі; новий стан — рядок у словнику.
6. Список можна змінити ззовні й дописати в історію будь-що; кортеж — незмінна копія.
7. `append` змінює об'єкт, а не присвоює ім'я; `+=` — присвоєння, і воно робить ім'я локальним.
8. У сервісі: це зв'язок між різними замовленнями, а замовлення про чужі доставки не знає.

</details>

### Шпаргалка

```python
self._bill                # внутрішнє — домовленість
self.__left               # → self._Promo__left, захист від колізій у нащадках

@property
def status(self):         # order.status — читання без дужок
    return self._status   # без сеттера — AttributeError при записі

@bill.setter
def bill(self, value):    # order.bill = ... — з перевіркою
    ...                   # спершу перевірки, потім зміна

TRANSITIONS = {"new": {"cooking", "cancelled"}, ...}   # автомат
def _move(self, new): ... # єдине місце зміни статусу

def make_counter():       # стан однієї дії — у замиканні
    count = 0
    def next_id():
        nonlocal count
        count += 1
        return count
    return next_id

__all__ = ["fare_for"]    # що віддає from module import *

# область видимості: локальна → замикання → атрибут об'єкта → модуль (лише константи); global — майже ніколи
# правило одного об'єкта — у моделі; правило між об'єктами — у сервісі
```

## Далі

- Практикум — [`lab_lesson_22_cars_oop.ipynb`](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_22_encapsulation_scope/lab_lesson_22_cars_oop.ipynb): «Автомобілі як об'єкти».
- **Урок 23 — практикум П4**: рекурсія, «розділяй і володарюй», перебір з поверненням.